# 06：在量子计算机上测量你的电路

**你将学到：**理想概率描述模型，真实硬件返回有限次测量的计数。一次 shot 是一次执行加一次测量。

我们将 Bell 电路提交到夸父（Quafu），并把测量计数与理想预测比较。

[English edition](../notebooks/06_quafu_hardware.ipynb)



## 1. 准备 workshop 环境

直接运行设置单元，无需修改。平台凭据会自动读取，不会在这里输入或显示。
保存的结果也可以用于扩展 notebook `06_compare_results`。

中英文版使用同一用户的个人结果目录和回执；换一种语言学习时，仍会恢复已经保存的任务，而不是重新提交。


In [ ]:
from pathlib import Path
import os
import sys
import json

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next((root / "flagos2026" for root in search_roots
                 if (root / "flagos2026/notebooks").is_dir()), None)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks. Open the deployed workshop first.")
sys.path.insert(0, str(WORKSHOP / "scripts"))
from cloud_lesson import workshop_directory

WORKSHOP = workshop_directory(Path.cwd())
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


## 2. 构造电路并检查理想答案

理想情况下，只出现 `00` 和 `11`，各自概率为一半。

电路图中，每条横线对应一个量子比特，量子门从左到右执行。图中的上下顺序不等于云平台结果位串的排列顺序。


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
ideal = fq.run(circuit)
print("Ideal probabilities:", ideal.state.abs().square().tolist())


## 3. 选择量子硬件

请和讲师确认当天可用的目标设备。一次 shot 是一次执行并测量。
**打开提交开关并运行提交单元，会创建一个测量 1024 次的真实 QPU 任务，使用你的额度。**


In [ ]:
target = os.getenv("QUAFU_TARGET", "quafu:Baihua")
shots = 1024
submit_to_hardware = False


### 准备任务恢复文件（直接运行，无需修改）

这个折叠单元防止已保存任务被意外重复提交。它会检查电路与配置，并用 `fq.restore_job()` 恢复匹配的回执；它本身从不提交任务。
存储与凭据属于环境准备，不是量子计算操作。


In [ ]:
from flagquantum.core.ir import ensure_circuit_ir

receipt_path = OUTPUTS / "quafu-bell.receipt.json"
guard_path = OUTPUTS / "quafu-bell.submission.json"
request = {
    "circuit_hash": ensure_circuit_ir(circuit).content_hash,
    "target": target,
    "shots": shots,
    "image": None,
    "project": None,
    "queue": None,
}
job = None
if submit_to_hardware:
    if not os.getenv("QUAFU_API_KEY"):
        raise RuntimeError("Configure Quafu access outside this notebook; nothing submitted.")
    if receipt_path.exists():
        saved = json.loads(guard_path.read_text())
        if any(saved.get(key) != value for key, value in request.items()):
            raise RuntimeError("Saved task belongs to another experiment. Keep its files and use a new receipt filename.")
        job = fq.restore_job(receipt_path)
    else:
        if (OUTPUTS / "quafu-bell.json").exists():
            raise RuntimeError("Existing hardware result found without a receipt. Keep it and reconcile the task before submitting.")
        try:
            with guard_path.open("x") as stream:
                json.dump(request, stream, indent=2)
        except FileExistsError as exc:
            raise RuntimeError("An earlier attempt has no receipt. Reconcile it on the platform; do not resubmit.") from exc


## 4. 通过 FlagQuantum 提交

`fq.submit()` 返回时不会等待任务执行完成。立即调用 `job.save()` 保存回执，就能在内核重启后恢复同一个任务。
重复运行此 notebook 时，会恢复记录中的任务，而不是再次提交。

如果要改变电路、目标或 shots 进行新实验，请为新的实验使用新的回执文件，不要把旧任务的结果当成新实验的结果。


In [ ]:
if submit_to_hardware and job is None:
    job = fq.submit(
        circuit, target=target, shots=shots, outputs=fq.counts(), compiler=None,
    )
    print("Task ID:", job.id)
    job.save(receipt_path)
elif job is not None:
    print("Restored task:", job.id)
else:
    print("Local rehearsal only. Enable the switch to submit a real task.")


## 5. 查询进度

这里执行一次只读查询。`queued` 表示排队，`running` 表示正在执行，`succeeded` 表示平台报告已完成。
结果交付是另一个环节，仍可能延迟或暂不可用。


In [ ]:
if job is not None:
    print("Status:", job.status())


## 6. 短暂等待并读取结果

`job.wait()` 将轮询状态和获取结果组合起来。超时不会停止任务，可以稍后重新运行此单元。
也可以在状态为 `succeeded` 后调用 `job.result()`。
如果远程任务失败，不会用本地结果冒充远程结果。


In [ ]:
result = None
if job is not None:
    try:
        result = job.wait(timeout=30)
    except TimeoutError:
        print("Result not available yet; last provider status:", job.raw_status)
        print("Rerun only this result cell later; do not resubmit. If completion is reported but results remain unavailable, ask the instructor to inspect task logs.")
if result is not None:
    counts = result.counts[0]
    assert sum(counts.values()) == shots
    print("Measured counts:", counts)
    print("Total shots:", sum(counts.values()))


### 保存结果，供扩展示例比较（可选）

这个折叠导出单元记录真实的任务 ID 和测量计数。理解或使用 `fq.submit()` 并不需要运行这一步。


In [ ]:
if result is not None:
    record = {
        "source": "live", "status": "completed", "target": target,
        "shots": shots, "task_id": job.id, "counts": counts,
    }
    (OUTPUTS / "quafu-bell.json").write_text(json.dumps(record, indent=2))


## 带回家使用

离开 Jupyter 后仍然使用相同的标准 API。这个简短示例每次被主动执行都会创建新的真实任务；请在源码之外配置访问凭据，并在等待之前保存回执。

```python
import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
job = fq.submit(circuit, target="quafu:Baihua", shots=1024, outputs=fq.counts())
job.save("bell.receipt.json")
print(job.id, job.status())
```

恢复已保存任务：`job = fq.restore_job("bell.receipt.json")`。完成后可查看 `job.result().counts`。

**解释结果：**为什么可能出现 `01` 和 `10`？增加 shots 能消除设备噪声吗？
结果比较可参考扩展 notebook `06_compare_results`，环境配置和恢复细节见[云任务说明](../CLOUD_JOBS.md)。
